# Milestone 3. Huấn luyện Citation Matching trên Kaggle

Gắn thư mục đã chuẩn hóa `kaggle_dataset` làm Kaggle Dataset. Gói này gồm `labels.parquet`, `references.parquet`, `partitions.csv`, `summary.json` và `m3_training_code.zip`. Notebook nhận cả file ZIP và thư mục code Kaggle đã giải nén. Trên máy local, mở notebook từ thư mục gốc của repo.

Notebook dùng CPU mặc định. Nếu đã bật GPU trong cài đặt Kaggle, đặt `USE_GPU = True` ở ô huấn luyện. Bật Internet nếu môi trường cần tải thư viện.

Logistic Regression và XGBoost được fit bằng nhãn tự động. Điểm valid và test dùng nhãn tay đã duyệt từ các bài nguồn riêng biệt.


In [ ]:
# Thiết lập môi trường
import json
import os
import subprocess
import sys
from importlib.metadata import version
from importlib.util import find_spec
from pathlib import Path
from zipfile import ZipFile

KAGGLE_INPUT = Path('/kaggle/input')
IS_KAGGLE = KAGGLE_INPUT.is_dir()
DATASET_DIR = None  # Điền một đường dẫn nếu đã gắn nhiều bộ dữ liệu.
if IS_KAGGLE:
    dataset_dir = Path(DATASET_DIR) if DATASET_DIR else None
    if dataset_dir is None:
        candidates = sorted(p.parent for p in KAGGLE_INPUT.rglob('labels.parquet')
                            if (p.parent / 'references.parquet').is_file()
                            and (p.parent / 'summary.json').is_file())
        if len(candidates) != 1:
            raise ValueError('Gắn bộ dữ liệu huấn luyện hoặc điền DATASET_DIR.')
        dataset_dir = candidates[0]
    code_archive = dataset_dir / 'm3_training_code.zip'
    if code_archive.is_file():
        repo_dir = Path('/kaggle/working/m3_training_code')
        repo_dir.mkdir(exist_ok=True)
        with ZipFile(code_archive) as archive:
            archive.extractall(repo_dir)
    else:
        source_roots = [p.parent.parent for p in dataset_dir.rglob('train_matching.py')
                        if p.parent.name == 'src' and (p.parent / 'requirements-m3.txt').is_file()]
        if len(source_roots) != 1:
            raise FileNotFoundError('Thiếu gói m3_training_code.zip hoặc thư mục src đã giải nén.')
        repo_dir = source_roots[0]
else:
    repo_dir = Path.cwd()
    dataset_dir = Path(DATASET_DIR) if DATASET_DIR else repo_dir / 'kaggle_dataset'

requirements = repo_dir / 'src' / 'requirements-m3.txt'
if not requirements.is_file():
    raise FileNotFoundError(f'Không tìm thấy code huấn luyện tại {repo_dir}.')
missing = any(find_spec(name) is None for name in ('numpy', 'pandas', 'pyarrow', 'sklearn', 'xgboost'))
if missing or int(version('xgboost').split('.', 1)[0]) < 2:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=True)


In [ ]:
# Đọc manifest của bộ dữ liệu
DATASET_DIR = dataset_dir.resolve()
OUTPUT_DIR = Path('/kaggle/working/outputs_m3') if IS_KAGGLE else repo_dir / 'outputs_m3' / 'audited'
summary = json.loads((DATASET_DIR / 'summary.json').read_text(encoding='utf-8'))
print('Dataset:', DATASET_DIR)
print('Output:', OUTPUT_DIR)
print('Số bài theo tập:', summary['partition_papers'])


In [ ]:
# Huấn luyện
USE_GPU = False
WORKERS = max(1, min(6, os.cpu_count() or 1))
command = [
    sys.executable, '-B', '-m', 'src.train_matching',
    '--dataset-dir', str(DATASET_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--workers', str(WORKERS),
]
if USE_GPU:
    command.append('--use-gpu')

subprocess.run(command, cwd=repo_dir, check=True)
results = json.loads((OUTPUT_DIR / 'metrics_summary.json').read_text(encoding='utf-8'))


In [ ]:
#@title 4. Tổng hợp bảng so sánh mô hình, trọng số Logistic Regression & Feature Importance XGBoost
import pandas as pd

rows = []
for model_name, splits in results['evaluation'].items():
    for split_name in ('valid', 'test'):
        for subset_name, m in splits[split_name].items():
            rows.append({
                'model': model_name,
                'split': split_name,
                'subset': subset_name,
                'queries': m['total_queries'],
                'pos_queries': m['positive_queries'],
                'positives_in_pool': m['positives_in_pool'],
                'missing_bibtex': m['queries_missing_bibtex'],
                'MRR@5': round(m['mrr@5'], 4) if m['mrr@5'] is not None else None,
                'Hit@1': round(m['hit@1'], 4) if m['hit@1'] is not None else None,
                'Hit@5': round(m['hit@5'], 4) if m['hit@5'] is not None else None,
                'E2E_Acc': round(m['e2e_accuracy'], 4) if 'e2e_accuracy' in m else None,
                'Matched_F1': round(m['matched_f1'], 4) if 'matched_f1' in m else None,
                'Threshold': round(m['threshold'], 3) if 'threshold' in m else None,
            })

display(pd.DataFrame(rows))

logreg = json.loads((OUTPUT_DIR / 'logreg_model.json').read_text(encoding='utf-8'))
print('Logistic Regression Weights (scaled & raw)')
display(pd.DataFrame({
    'feature': logreg['features'],
    'scaled_coef': logreg['coefficients'],
    'raw_coef': logreg['raw_coefficients'],
}))
print('Intercept (scaled):', logreg['intercept'], '| Intercept (raw):', logreg['raw_intercept'])

xgb_imp_path = OUTPUT_DIR / 'xgboost_feature_importances.json'
if xgb_imp_path.exists():
    xgb_imp = json.loads(xgb_imp_path.read_text(encoding='utf-8'))
    print(f"XGBoost Feature Importances (best_iteration={xgb_imp['best_iteration']})")
    display(pd.DataFrame({
        'feature': xgb_imp['features'],
        'gain': [xgb_imp['gain_importance'][f] for f in xgb_imp['features']],
        'split_weight': [xgb_imp['split_weight_importance'][f] for f in xgb_imp['features']],
        'normalized': [xgb_imp['sklearn_normalized_importance'][f] for f in xgb_imp['features']],
    }).sort_values('gain', ascending=False))